In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.1 KV caching

During generation, the keys and values of past tokens never change, yet naive
decoding recomputes them every step. A **KV cache** stores them, so each new token
is O(1) extra work instead of O(T). The output is identical, that parity is the
thing to prove.

In [ ]:
from pocketlm import scaled_dot_product_attention, KVCache, cached_step

torch.manual_seed(0)
B, H, T, hs = 1, 2, 6, 8
q = torch.randn(B, H, T, hs)
k = torch.randn(B, H, T, hs)
v = torch.randn(B, H, T, hs)
full, _ = scaled_dot_product_attention(q, k, v, causal=True)
cache = KVCache()
outs = [cached_step(q[:, :, t:t + 1], k[:, :, t:t + 1], v[:, :, t:t + 1], cache)
        for t in range(T)]
incremental = torch.cat(outs, dim=2)
print("cached == full attention:", torch.allclose(incremental, full, atol=1e-5))

Step-by-step cached attention reproduces full causal attention exactly. The cache
trades a little memory for a large compute saving, the core inference optimization.

In [ ]:
assert torch.allclose(incremental, full, atol=1e-5)